# DermaReal · Experiment E2 — Real-image classification

This notebook builds and evaluates a **six-class facial skin-condition classifier using only real images**. It is the canonical real-data experiment for the manuscript. Its frozen real-image train/validation/test assignment is designed to be reused in Experiment E3 (real + synthetic training).

**Dataset:** `Facial Skin Conditions – Real Image Collection`, Kaggle Version 2, containing the folders `Acne`, `Fungal_infection`, `Hyper_Pigmentation`, `Normal_Skin`, `Skin_cancer`, and `Vitiligo`.

**Models:** EfficientNetV2-B0, ResNet-50, DenseNet-121, and MobileNetV3-Large, each trained in three stages (classification-head warm-up, partial unfreezing, deeper fine-tuning). Validation macro-F1 selects the best checkpoint. A validation-weighted soft-voting ensemble is evaluated on the held-out **real** test set.

**Workflow:** dataset inventory and checks → exact-content hashes → visual grouping → frozen group-aware split → independent similarity audit → shortcut controls → training → test metrics and ensemble → GitHub evidence archive.

**Reproducibility:** The notebook saves versioned config, image provenance/status, duplicate/group records, the frozen split, model checkpoints, epoch logs, per-image predictions, accuracy/precision/recall/F1/AUC/MCC, confusion matrices, figures, and an evidence ZIP. See the final cell for the artifact inventory.

**Before starting GPU training:** Inspect the label-review and duplicate-conflict reports. Filenames and visual hashes cannot establish a clinical diagnosis, and visual groups do not prove patient identity. Check the original image licences before publishing any images. The notebook excludes review-image galleries and model weights from the GitHub evidence ZIP.


## 0 · Install dependencies and initialize reproducibility

Checks core Python packages and imports the analysis and deep-learning libraries. Seeds NumPy, Python, and TensorFlow; records the local TensorFlow version and available GPU. Image hashes use a SciPy-based, explicitly versioned pHash implementation, so the notebook does not require downloading `ImageHash`.

**Output:** Dependency readiness and accelerator information in the notebook log.


In [ ]:
import importlib.util, subprocess, sys
for package, module in [('scikit-learn','sklearn'), ('Pillow','PIL'), ('seaborn','seaborn'), ('SciPy','scipy')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
print('Dependencies checked.')

In [ ]:
import os, sys, json, gc, time, hashlib, random, shutil, platform, logging, math
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image, ImageOps, ImageStat
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.fftpack import dct
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve, cohen_kappa_score,
    matthews_corrcoef)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2B0, ResNet50, DenseNet121, MobileNetV3Large

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('Note: deterministic TF kernels not available:', e)
print('Python:',sys.version.split()[0], 'TF:',tf.__version__, 'GPU:',tf.config.list_physical_devices('GPU'))

## 1 · Experiment configuration and output folders

Defines the exact six-class folder-to-label mapping, Kaggle input location, training epochs and batch size, similarity thresholds, and output directories. Records environment details and run parameters so the saved results can be associated with the code that produced them.

**Outputs:** `config/run_config.json`, `config/environment.json`, and `logs/run.log`.


In [ ]:
INPUT_HINT = Path(os.environ.get('REAL_DATA_DIR', '/kaggle/input'))
EXPECTED_FOLDERS = {
    'Acne':'Acne',
    'Fungal_infection':'Fungal_Infection',
    'Hyper_Pigmentation':'Hyperpigmentation',
    'Normal_Skin':'Normal_Skin',
    'Skin_cancer':'Skin_Cancer',
    'Vitiligo':'Vitiligo',
}
CLASSES = ['Acne','Fungal_Infection','Hyperpigmentation','Normal_Skin','Skin_Cancer','Vitiligo']
CLASS_TO_ID = {c:i for i,c in enumerate(CLASSES)}
VALID_EXTS = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff'}
ROOT = Path('/kaggle/working/dermareal_e2_run001') if Path('/kaggle/working').exists() else Path('dermareal_e2_run001')
DIRS = {name:ROOT/name for name in ['config','dataset','audits','splits','training','metrics','ensemble','figures','logs','models']}
for d in DIRS.values(): d.mkdir(parents=True,exist_ok=True)

# Set only if a CSV is available: required: relative_path; optional: original_diagnosis,
# source_dataset, source_url, patient_id, review_status, license, allowed_for_training.
SOURCE_MANIFEST_CSV = None
# For a later independent E3 run, point this to E2's splits_v3.csv; otherwise build once.
EXISTING_SPLIT_CSV = None
ALLOW_SUSPECT_FILENAMES = False  # change only AFTER manually reviewing flagged images
REQUIRE_PROVENANCE = False  # True to enforce source/license data for every image
RUN_SHORTCUT_CONTROLS = True
RUN_MODELS = ['EfficientNetV2B0','ResNet50','DenseNet121','MobileNetV3Large']
IMG_SIZE, BATCH_SIZE = 224, 24
P1_EPOCHS, P2_EPOCHS, P3_EPOCHS = 15,25,30
LR_BY_PHASE = [1e-3,1e-4,2e-5]
L2_REG, DROPOUT, FOCAL_GAMMA, FOCAL_ALPHA = 1e-4,0.4,2.0,0.25
PHASH_MAX_DIST, SIM_TAU_START, SIM_TAU_MAX, SIM_TAU_STEP = 5,0.90,0.98,0.02
MAX_GROUP_SHARE, SPLIT_TRIES, CHUNK = 0.02,200,192

logger = logging.getLogger('dermareal'); logger.setLevel(logging.INFO); logger.propagate=False
if not logger.handlers:
    h=logging.FileHandler(DIRS['logs']/'run.log',encoding='utf-8'); h.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s')); logger.addHandler(h)
    s=logging.StreamHandler(sys.stdout); s.setFormatter(logging.Formatter('%(levelname)s %(message)s')); logger.addHandler(s)

def json_save(obj,path):
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    with open(path,'w',encoding='utf-8') as f: json.dump(obj,f,indent=2,default=str,allow_nan=False)

config={'dataset_hint':str(INPUT_HINT),'classes':CLASSES,'seed':SEED,'size':IMG_SIZE,
        'batch_size':BATCH_SIZE,'epochs':[P1_EPOCHS,P2_EPOCHS,P3_EPOCHS],
        'learning_rates':LR_BY_PHASE,'focal_gamma':FOCAL_GAMMA,'focal_alpha':FOCAL_ALPHA,
        'phash_distance':PHASH_MAX_DIST,'tau_start':SIM_TAU_START,'tau_max':SIM_TAU_MAX,
        'max_group_share':MAX_GROUP_SHARE,'split_tries':SPLIT_TRIES,'models':RUN_MODELS,
        'dataset_label_status':'user_filtered_not_independently_clinically_verified',
        'phash_algorithm':'dct_32x32_gray_8x8_median_v1'}
json_save(config,DIRS['config']/'run_config.json')
json_save({'python':sys.version,'tensorflow':tf.__version__,'numpy':np.__version__,
           'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__,
           'PIL':__import__('PIL').__version__,'gpu':[str(x) for x in tf.config.list_physical_devices('GPU')],
           'platform':platform.platform()},DIRS['config']/'environment.json')

def find_data_root():
    bases=[INPUT_HINT]
    if INPUT_HINT.is_dir():
        bases += sorted(p for p in INPUT_HINT.rglob('Face_Skin_datset') if p.is_dir())
        bases += sorted(p for p in INPUT_HINT.rglob('Face_Skin_dataset') if p.is_dir())
    for base in bases:
        for folder in [base/'Face_Skin_datset',base/'Face_Skin_dataset',base]:
            if folder.is_dir() and set(EXPECTED_FOLDERS) == {x.name for x in folder.iterdir() if x.is_dir()}:
                return folder
    raise FileNotFoundError('Cannot find exactly six class folders. Check Kaggle Add Input and INPUT_HINT. Looked in: '+str(bases))

DATA_DIR=find_data_root()
logger.info('DATA_DIR=%s',DATA_DIR)
logger.info('OUTPUT_ROOT=%s',ROOT)

## 2 · Real-image inventory, decoding, and source metadata

Recursively reads every eligible image, checks that Pillow can fully decode it, measures image dimensions and basic image statistics, and computes whole-file **SHA-256 and MD5** checksums. Known suspicious filenames are listed for label review. Optional original-source metadata can be joined by relative file path; known rejected labels or disallowed images are excluded when the metadata provides those decisions.

**Outputs:** `dataset/dataset_manifest.csv`, `dataset/dataset_statistics.csv`, `dataset/invalid_images.csv`, `dataset/suspect_filename_labels.csv`, and `dataset/dataset_fingerprint.json`.

The actual diagnosis and training permissions must be established from original source records or a qualified review, not inferred from image appearance alone.


In [ ]:
SUSPECT = {
    'Skin_Cancer':['sjs-ten','sjs_ten','stevens-johnson','toxic-epidermal','toxic_epidermal'],
    'Normal_Skin':['rosacea','dermatitis','eczema','perioral','seborrheic','psoriasis','vitiligo','melanoma','carcinoma']}
records=[]; invalid=[]; suspicious=[]; total_candidates=0
for folder,cls in EXPECTED_FOLDERS.items():
    files=sorted(p for p in (DATA_DIR/folder).rglob('*') if p.is_file() and p.suffix.lower() in VALID_EXTS)
    total_candidates+=len(files)
    logger.info('%s: %s candidate files',folder,len(files))
    for path in files:
        relative=path.relative_to(DATA_DIR).as_posix()
        try:
            sha=hashlib.sha256(); md5=hashlib.md5()
            with path.open('rb') as f:
                for block in iter(lambda:f.read(1<<20),b''):
                    sha.update(block); md5.update(block)
            with Image.open(path) as img:
                img.load(); w,h=img.size
                if min(w,h)<64: raise ValueError('image smaller than 64 pixels')
                img=ImageOps.exif_transpose(img).convert('RGB')
                small=np.array(img.resize((64,64),Image.Resampling.BILINEAR),dtype=np.float32)
                std=float(small.std()); dark=float((small<4).all(axis=2).mean())
                if std<3 or dark>0.97: raise ValueError('blank/black image')
                border=np.concatenate([small[:8].reshape(-1,3),small[-8:].reshape(-1,3),
                    small[:, :8].reshape(-1,3),small[:, -8:].reshape(-1,3)],axis=0)
                bmean=border.mean(axis=0); bstd=border.std(axis=0)
            item={'relative_path':relative,'path':str(path),'folder':folder,'class':cls,'label':CLASS_TO_ID[cls],
                  'sha256':sha.hexdigest(),'md5':md5.hexdigest(),'width':w,'height':h,'aspect_ratio':round(w/h,6),
                  'size_bytes':path.stat().st_size,'format':path.suffix.lower(),
                  'pixel_std':round(std,3),'dark_fraction':round(dark,4),
                  **{f'border_mean_{k}':float(bmean[i]) for i,k in enumerate('rgb')},
                  **{f'border_std_{k}':float(bstd[i]) for i,k in enumerate('rgb')}}
            records.append(item)
            matches=[word for word in SUSPECT.get(cls,[]) if word in path.name.lower()]
            if matches: suspicious.append({'relative_path':relative,'class':cls,'matched_tokens':';'.join(matches)})
        except Exception as e: invalid.append({'relative_path':relative,'reason':repr(e)})

manifest=pd.DataFrame(records).sort_values('relative_path').reset_index(drop=True)
assert len(manifest)>0, 'No usable images found'
pd.DataFrame(invalid,columns=['relative_path','reason']).to_csv(DIRS['dataset']/'invalid_images.csv',index=False)
pd.DataFrame(suspicious,columns=['relative_path','class','matched_tokens']).to_csv(DIRS['dataset']/'suspect_filename_labels.csv',index=False)

decoded_before_provenance=len(manifest)
# Optional fully traceable provenance. NEVER guess patient IDs or original labels.
manifest['source_dataset']='unknown'; manifest['source_url']=''; manifest['patient_id']=''
manifest['original_diagnosis']=''; manifest['review_status']='unverified'
manifest['license']='unknown'; manifest['allowed_for_training']='unknown'
if SOURCE_MANIFEST_CSV:
    origin=pd.read_csv(SOURCE_MANIFEST_CSV,dtype=str).fillna('')
    assert 'relative_path' in origin.columns and origin.relative_path.is_unique
    fields=['source_dataset','source_url','patient_id','original_diagnosis','review_status','license','allowed_for_training']
    for name in fields:
        if name in origin:
            value_map=origin.set_index('relative_path')[name]
            manifest[name]=manifest.relative_path.map(value_map).fillna('')
    reject=manifest.allowed_for_training.str.lower().isin(['false','no','0','not_allowed'])
    reject|=manifest.review_status.str.lower().isin(['reject','rejected','incorrect_label'])
    if reject.any():
        manifest.loc[reject,['relative_path','class','sha256','md5','source_dataset',
            'original_diagnosis','review_status','license','allowed_for_training']].to_csv(
                DIRS['dataset']/'provenance_excluded.csv',index=False)
        manifest=manifest.loc[~reject].copy().reset_index(drop=True)
    logger.info('Provenance manifest merged: %s',SOURCE_MANIFEST_CSV)

provenance_removed=decoded_before_provenance-len(manifest)
if not (DIRS['dataset']/'provenance_excluded.csv').exists():
    pd.DataFrame(columns=['relative_path','class','sha256','md5','source_dataset',
        'original_diagnosis','review_status','license','allowed_for_training']).to_csv(
        DIRS['dataset']/'provenance_excluded.csv',index=False)
# Patient identifiers must NOT be exported to a public research repository.
# Use a private ephemeral HMAC key (or DERMA_PATIENT_HMAC_KEY from Kaggle Secrets).
# Raw IDs are discarded; equality is retained for grouping within this run.
import hmac, secrets
_private_patient_key = os.environ.get('DERMA_PATIENT_HMAC_KEY', '').encode('utf-8') or secrets.token_bytes(32)
manifest['patient_id'] = [
    hmac.new(_private_patient_key, (str(src)+'|'+str(pid)).encode('utf-8'), hashlib.sha256).hexdigest()[:32]
    if str(pid).strip() and str(pid).lower() != 'nan' else ''
    for src,pid in zip(manifest['source_dataset'],manifest['patient_id'])
]
del _private_patient_key

suspicious_active=[x for x in suspicious if x['relative_path'] in set(manifest.relative_path)]
manifest['status']='candidate'
manifest.to_csv(DIRS['dataset']/'dataset_manifest.csv',index=False)
counts=manifest.groupby('class').size().reindex(CLASSES,fill_value=0).rename('usable_count').reset_index()
counts.to_csv(DIRS['dataset']/'dataset_statistics.csv',index=False)
logger.info('Usable image counts:\n%s',counts.to_string(index=False))
logger.info('Invalid=%s; suspicious filenames=%s; clinical labels NOT automatically validated',len(invalid),len(suspicious))
if REQUIRE_PROVENANCE:
    assert (manifest.source_dataset.fillna('').str.strip().ne('') & manifest.source_dataset.ne('unknown')).all() and (manifest.license.fillna('').str.strip().ne('') & manifest.license.ne('unknown')).all(), 'Missing provenance/licence'
if suspicious_active and not ALLOW_SUSPECT_FILENAMES:
    raise RuntimeError('Known wrong-label filename patterns detected. Review suspect_filename_labels.csv; correct/remove the files, then rerun. Override only after documented verification.')
assert (counts.usable_count>5).all(), 'A class has too few valid files.'

# Dataset content fingerprint: different data cannot silently reuse a split.
dataset_fingerprint=hashlib.sha256('\n'.join(manifest.relative_path+'|'+manifest.sha256+'|'+manifest['class']).encode()).hexdigest()
json_save({'fingerprint_sha256':dataset_fingerprint,'files':len(manifest),'invalid':len(invalid),
           'total_candidate_images':total_candidates,'valid_decoded_images':decoded_before_provenance,
           'removed_by_provenance':provenance_removed,
           'suspect_filename_count':len(suspicious), 'suspect_filename_still_active':len(suspicious_active),'clinical_label_verification':'NOT INFERRED'},
          DIRS['dataset']/'dataset_fingerprint.json')
logger.info('Dataset fingerprint: %s',dataset_fingerprint)
json_save({'candidate_files':total_candidates,'invalid_images':len(invalid),
           'decoded_images':decoded_before_provenance,'provenance_exclusions':provenance_removed,
           'eligible_for_grouping':len(manifest),'suspicious_active_filenames':len(suspicious_active)},
          DIRS['dataset']/'validation_summary.json')

## 3 · Exact duplicates, pHash, embeddings, and visual groups

Finds byte-identical files using SHA-256 (with MD5 as a secondary audit), computes orientation-aware 64-bit perceptual hashes, and links visually similar images using ImageNet ResNet-50 embeddings. Union-Find combines these links and any provided source-qualified patient IDs into **visual groups**. A small number of conflicting-label groups may need manual inspection; the notebook quarantines them rather than allowing them to cross splits. Identical copies are removed without discarding every visually related photograph.

**Outputs:** `audits/exact_sha256_groups.csv`, `audits/exact_md5_groups.csv`, `audits/group_membership.csv`, `audits/group_summary.csv`, `audits/grouping_summary.json`, `audits/cross_class_conflicts.csv`, `audits/excluded_images.csv`, plus class/source removal summaries. Private review galleries are stored in `private_review/` and never added to the GitHub archive.


In [ ]:
class UnionFind:
    def __init__(self,n): self.parent=np.arange(n,dtype=np.int32); self.rank=np.zeros(n,dtype=np.int8)
    def find(self,x):
        p=self.parent
        while p[x]!=x: p[x]=p[p[x]]; x=p[x]
        return int(x)
    def union(self,x,y):
        a,b=self.find(int(x)),self.find(int(y))
        if a==b: return
        if self.rank[a]<self.rank[b]: a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]: self.rank[a]+=1
    def groups(self):
        roots=np.array([self.find(i) for i in range(len(self.parent))])
        return pd.factorize(roots,sort=True)[0].astype(np.int32)

N=len(manifest); uf_exact=UnionFind(N)
for _,indices in manifest.groupby('sha256').indices.items():
    ii=list(indices)
    for j in ii[1:]: uf_exact.union(ii[0],j)
manifest['exact_group']=uf_exact.groups()
exact_g=manifest.groupby('sha256').agg(image_count=('path','size'),n_classes=('class','nunique'))
exact_g.to_csv(DIRS['audits']/'exact_sha256_groups.csv')
md5_g=manifest.groupby('md5').agg(image_count=('path','size'),distinct_sha256=('sha256','nunique'),n_classes=('class','nunique'))
md5_g.to_csv(DIRS['audits']/'exact_md5_groups.csv')
if (md5_g.distinct_sha256>1).any():
    raise RuntimeError('MD5 collision or inconsistent hashing detected; investigate before splitting')
logger.info('Exact content: %s repeated-hash groups, %s extra copies, %s conflicting hashes',
            int((exact_g.image_count>1).sum()),int((exact_g.image_count-1).sum()),int((exact_g.n_classes>1).sum()))

from PIL import ImageOps

def phash64(image):
    # Equivalent 32x32 DCT / 8x8 median algorithm to ImageHash.phash(hash_size=8).
    gray=image.convert('L').resize((32,32),Image.Resampling.LANCZOS)
    pixels=np.asarray(gray,dtype=np.float32)
    coeff=dct(dct(pixels,axis=0),axis=1)
    lo=coeff[:8,:8]; bits=(lo>np.median(lo)).reshape(-1)
    value=0
    for bit in bits: value=(value<<1)|int(bit)
    return value

def hash_variants(path):
    with Image.open(path) as raw:
        im=ImageOps.exif_transpose(raw).convert('RGB')
        variants=[im,ImageOps.mirror(im),ImageOps.flip(im),ImageOps.mirror(ImageOps.flip(im))]
        return [phash64(a) for a in variants]

HV=np.asarray([hash_variants(p) for p in manifest.path],dtype=np.uint64)
manifest['phash_original']=[f'{int(x):016x}' for x in HV[:,0]]
POP=np.array([int(i).bit_count() for i in range(256)],dtype=np.uint8)
def hamming_block(a,b):
    xor=np.bitwise_xor(a[:,None],b[None,:])
    bytes_=np.ascontiguousarray(xor).view(np.uint8).reshape(len(a),len(b),8)
    return POP[bytes_].sum(axis=2,dtype=np.uint8)

uf_phash=UnionFind(N)
plain_pairs=0; flipped_pairs=0
for start in range(0,N,CHUNK):
    stop=min(N,start+CHUNK)
    d0=hamming_block(HV[start:stop,0],HV[:,0]); d=d0.copy()
    for k in range(1,4):
        np.minimum(d,hamming_block(HV[start:stop,0],HV[:,k]),out=d)
        np.minimum(d,hamming_block(HV[start:stop,k],HV[:,0]),out=d)
    row,col=np.nonzero(d<=PHASH_MAX_DIST); row=row+start
    keep=col>row; row,col=row[keep],col[keep]
    for a,b in zip(row,col): uf_phash.union(a,b)
    flipped_pairs+=len(row)
    row2,col2=np.nonzero(d0<=PHASH_MAX_DIST); row2=row2+start
    plain_pairs+=int((col2>row2).sum())
manifest['phash_group']=uf_phash.groups()
logger.info('pHash: plain linked pairs %d; flip-aware %d; flip-aware groups %d',plain_pairs,flipped_pairs,manifest.phash_group.nunique())

# Embed images with same aspect-preserving letterbox used in the training pipeline.
def load_rgb(path,size=224):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im=ImageOps.pad(im,(size,size),method=Image.Resampling.BILINEAR,color=(127,127,127))
        return np.asarray(im,dtype=np.float32)

def embed_images(model_name, batch_size=32):
    if model_name=='resnet':
        net=ResNet50(include_top=False,weights='imagenet',pooling='avg',input_shape=(IMG_SIZE,IMG_SIZE,3))
    else:
        net=EfficientNetV2B0(include_top=False,weights='imagenet',pooling='avg',
                             input_shape=(IMG_SIZE,IMG_SIZE,3),include_preprocessing=True)
    a,b=[],[]
    for start in range(0,N,batch_size):
        arr=np.stack([load_rgb(p,IMG_SIZE) for p in manifest.path.iloc[start:start+batch_size]])
        flipped=arr[:,:,::-1,:].copy()
        if model_name=='resnet':
            arr=keras.applications.resnet50.preprocess_input(arr)
            flipped=keras.applications.resnet50.preprocess_input(flipped)
        a.append(net.predict(arr,verbose=0)); b.append(net.predict(flipped,verbose=0))
    del net; gc.collect(); keras.backend.clear_session()
    return np.concatenate(a).astype(np.float32),np.concatenate(b).astype(np.float32)

def normalized_embeddings(a,b):
    mu=a.mean(axis=0,keepdims=True)
    a=a-mu; b=b-mu
    a/=np.linalg.norm(a,axis=1,keepdims=True)+1e-8
    b/=np.linalg.norm(b,axis=1,keepdims=True)+1e-8
    return a,b

logger.info('Calculating ResNet-50 grouping embeddings for %s images',N)
E,Ef=normalized_embeddings(*embed_images('resnet'))
# Candidate links; store similarities so tau can be tightened without re-embedding.
ei,ej,es=[],[],[]
for start in range(0,N,CHUNK):
    sim=np.maximum(E[start:start+CHUNK]@E.T,E[start:start+CHUNK]@Ef.T)
    ii,jj=np.nonzero(sim>=SIM_TAU_START); ii=ii+start
    take=jj>ii
    ii,jj=ii[take],jj[take]
    if len(ii): ei.append(ii.astype(np.int32));ej.append(jj.astype(np.int32));es.append(sim[(ii-start),jj].astype(np.float32))
ei=np.concatenate(ei) if ei else np.array([],dtype=np.int32)
ej=np.concatenate(ej) if ej else np.array([],dtype=np.int32)
es=np.concatenate(es) if es else np.array([],dtype=np.float32)
del E,Ef; gc.collect()

# Base pHash grouping + exact file hashes + real patient IDs (if present).
def make_groups(tau):
    u=UnionFind(N)
    for col in ['phash_group','exact_group']:
        for _,inds in manifest.groupby(col).indices.items():
            ind=list(inds)
            for j in ind[1:]:u.union(ind[0],j)
    known=manifest[manifest.patient_id.fillna('').str.strip().ne('') &
                   manifest.source_dataset.fillna('').str.strip().ne('unknown')]
    for _,inds in known.groupby(['source_dataset','patient_id']).indices.items():
        ind=list(known.index[inds])
        for j in ind[1:]:u.union(ind[0],j)
    for x,y in zip(ei[es>=tau],ej[es>=tau]):u.union(x,y)
    return u.groups()

tau=SIM_TAU_START
while True:
    groups=make_groups(tau)
    group_sizes=pd.Series(groups).value_counts()
    if group_sizes.max()/N<=MAX_GROUP_SHARE or tau>=SIM_TAU_MAX-1e-8: break
    tau=round(min(tau+SIM_TAU_STEP,SIM_TAU_MAX),2)
manifest['group']=groups
logger.info('Final groups=%d; tau=%.2f; largest group=%d (%.2f%%)',
            manifest.group.nunique(),tau,group_sizes.max(),100*group_sizes.max()/N)
if group_sizes.max()/N>MAX_GROUP_SHARE:
    logger.warning('Largest group exceeds group-share target; inspect group_summary.csv')

stats=manifest.groupby('group').agg(group_size=('path','size'),n_classes=('class','nunique'),
                                    classes=('class',lambda s:';'.join(sorted(s.unique()))),
                                    distinct_sha=('sha256','nunique'))
stats.to_csv(DIRS['audits']/'group_summary.csv')
conflict_ids=stats.index[stats.n_classes>1].tolist()
manifest.loc[manifest.group.isin(conflict_ids)].to_csv(DIRS['audits']/'cross_class_conflicts.csv',index=False)
logger.warning('%d cross-class visual groups quarantined pending manual review',len(conflict_ids))
# Quarantine ALL members of conflicting-class groups to avoid label-based leakage.
manifest['status']='eligible';manifest.loc[manifest.group.isin(conflict_ids),'status']='quarantined_cross_class'
# Keep ONE identical-file copy per class. All same-group related but non-identical images are retained.
dup_mask=(manifest.status=='eligible') & manifest.duplicated('sha256',keep='first')
manifest.loc[dup_mask,'status']='removed_exact_duplicate'
manifest.to_csv(DIRS['dataset']/'dataset_manifest.csv',index=False)
manifest.to_csv(DIRS['audits']/'group_membership.csv',index=False)
manifest.loc[manifest.status!='eligible'].to_csv(DIRS['audits']/'excluded_images.csv',index=False)
clean=manifest.loc[manifest.status=='eligible'].copy().reset_index(drop=True)
assert clean.groupby('class').size().reindex(CLASSES,fill_value=0).min()>10,'A class too small after quarantine'
report={'original_files':total_candidates,'valid_after_decode':decoded_before_provenance,
        'after_provenance_exclusions':N,'invalid_files':len(invalid),
        'provenance_excluded':provenance_removed,'final_eligible':len(clean),
        'exact_hash_groups_repeated':int((exact_g.image_count>1).sum()),
        'extra_exact_copies_total':int((exact_g.image_count-1).sum()),
        'exact_duplicate_copies_removed':int(dup_mask.sum()),
        'cross_class_groups_quarantined':len(conflict_ids),
        'cross_class_images_quarantined':int((manifest.status=='quarantined_cross_class').sum()),
        'unique_visual_groups':int(manifest.group.nunique()),
        'eligible_visual_groups':int(clean.group.nunique()),
        'multi_image_groups':int((stats.group_size>1).sum()),
        'largest_group':int(stats.group_size.max()),
        'plain_phash_pairs':plain_pairs,'flip_aware_phash_pairs':flipped_pairs,
        'embedding_tau':tau,'patient_grouping_used':bool((manifest.patient_id!='').any()),
        'group_type':'visual similarity; known patient ID when supplied; not inferred patient identity'}
json_save(report,DIRS['audits']/'grouping_summary.json')
class_stats=manifest.groupby(['class','status']).size().rename('count').reset_index()
class_stats.to_csv(DIRS['audits']/'removals_by_class_reason.csv',index=False)
manifest.groupby(['source_dataset','status'],dropna=False).size().rename('image_count').reset_index().to_csv(
    DIRS['audits']/'removals_by_source_reason.csv',index=False)
manifest.loc[manifest.status!='eligible',
    ['relative_path','class','source_dataset','sha256','md5','group','status']].to_csv(
    DIRS['audits']/'excluded_image_details.csv',index=False)
# PRIVATE gallery for diagnosis review; this may include copyrighted or identifiable images.
# It is deliberately kept OUT of the GitHub evidence ZIP.
PRIVATE_REVIEW_DIR=ROOT/'private_review'
PRIVATE_REVIEW_DIR.mkdir(parents=True,exist_ok=True)
if conflict_ids:
    sample_ids=conflict_ids[:min(20,len(conflict_ids))]
    for group_id in sample_ids:
        group_rows=manifest.loc[manifest.group==group_id].head(8)
        fig,axes=plt.subplots(2,4,figsize=(12,7))
        for ax in axes.ravel():ax.axis('off')
        for ax,(_,rec) in zip(axes.ravel(),group_rows.iterrows()):
            ax.imshow(load_rgb(rec['path'],160).astype(np.uint8))
            ax.set_title(f"{rec['class']}\n{Path(rec['path']).name[:30]}",fontsize=7)
        fig.suptitle(f'PRIVATE — conflicting visual group {group_id}: manual review required')
        fig.tight_layout()
        fig.savefig(PRIVATE_REVIEW_DIR/f'conflict_group_{group_id}.png',dpi=110)
        plt.close(fig)
print(json.dumps(report,indent=2))

## 4 · Frozen group-aware train/validation/test split

Creates a seeded, approximately **70% / 15% / 15%** partition by complete visual group rather than individual image. Validates that SHA-256 hashes and group identifiers do not cross partitions, and that all six classes appear in each partition. The split includes the dataset fingerprint and relative paths for reuse across Kaggle mounts.

**Outputs:** `splits/splits_v3.csv`, `splits/split_manifest.csv`, `splits/split_class_counts.csv`, `splits/split_group_counts.csv`, `splits/split_assertions.json`, `figures/fig1_corrected_dataset_distribution.png`, and `figures/fig4_real_split.png`.

For E3, reuse these exact real validation and test assignments rather than generating another real split.


In [ ]:
def best_group_split(y,g,test_fraction,tries,seed):
    y=np.asarray(y,dtype=int);g=np.asarray(g)
    target=np.bincount(y,minlength=len(CLASSES))/len(y)
    best=None
    for i in range(tries):
        splitter=GroupShuffleSplit(n_splits=1,test_size=test_fraction,random_state=seed+i)
        a,b=next(splitter.split(np.zeros(len(y)),y,groups=g))
        if not len(a) or not len(b):continue
        counts=np.bincount(y[b],minlength=len(CLASSES)); prop=counts/len(b)
        score=float(np.max(np.abs(prop-target)/(target+1e-8))+2*abs(len(b)/len(y)-test_fraction))
        if best is None or score<best[0]:best=(score,a,b)
    if best is None:raise ValueError('Group split failed; insufficient independent groups')
    return best[1],best[2]

if EXISTING_SPLIT_CSV:
    old=pd.read_csv(EXISTING_SPLIT_CSV,dtype={'sha256':str,'relative_path':str,'class':str,'split':str})
    assert set(old.columns)>={'relative_path','sha256','class','split','group','dataset_fingerprint'}
    assert old.relative_path.is_unique and set(old.relative_path)==set(clean.relative_path)
    assert old['split'].isin(['train','val','test']).all()
    assert set(old.dataset_fingerprint)=={dataset_fingerprint},'Dataset fingerprint mismatch: do not reuse split'
    tmp=clean[['relative_path','sha256','class','group']].merge(old[['relative_path','sha256','class','split']],on='relative_path',suffixes=('_actual','_saved'),validate='one_to_one')
    assert (tmp.sha256_actual==tmp.sha256_saved).all() and (tmp.class_actual==tmp.class_saved).all()
    clean['split']=clean.relative_path.map(old.set_index('relative_path')['split'])
    logger.info('Reusing strictly checked existing split: %s',EXISTING_SPLIT_CSV)
else:
    tr,temp=best_group_split(clean.label,clean.group,0.30,SPLIT_TRIES,SEED)
    va_sub,te_sub=best_group_split(clean.label.iloc[temp],clean.group.iloc[temp],0.50,SPLIT_TRIES,SEED+3000)
    split=np.empty(len(clean),dtype=object)
    split[tr]='train'; split[temp[va_sub]]='val'; split[temp[te_sub]]='test'
    clean['split']=split

assert clean.split.isin(['train','val','test']).all()
assert clean.groupby('group').split.nunique().max()==1,'Group leakage!'
assert clean.groupby('sha256').split.nunique().max()==1,'SHA-256 leakage!'
assert all(clean.loc[clean.split==s,'class'].nunique()==len(CLASSES) for s in ['train','val','test']), 'A split lacks one of six classes'
clean['dataset_fingerprint']=dataset_fingerprint
export_cols=['relative_path','sha256','class','label','group','split','dataset_fingerprint',
             'source_dataset','patient_id','review_status']
clean[export_cols].to_csv(DIRS['splits']/'splits_v3.csv',index=False)
# Preserve absolute mount path separately for this run; E3 should remap on relative_path, not on mount prefix.
clean.to_csv(DIRS['splits']/'split_manifest.csv',index=False)
counts=pd.crosstab(clean['class'],clean['split']).reindex(index=CLASSES,columns=['train','val','test'],fill_value=0)
counts.to_csv(DIRS['splits']/'split_class_counts.csv')
ng=clean.groupby(['split','class']).group.nunique().rename('visual_groups').reset_index()
ng.to_csv(DIRS['splits']/'split_group_counts.csv',index=False)
logger.info('Frozen counts:\n%s',counts.to_string())
json_save({'fingerprint':dataset_fingerprint,'group_overlap':0,'sha256_overlap':0,
           'split_counts':clean.split.value_counts().to_dict(),
           'split_csv':'splits/splits_v3.csv'},DIRS['splits']/'split_assertions.json')

# Publication figure 1 & figure 4 are newly generated from THIS dataset, not copied from old run.
fig,ax=plt.subplots(figsize=(10,4)); counts.sum(axis=1).plot.bar(ax=ax,rot=25)
ax.set_title('Corrected real facial skin dataset: usable images by class');ax.set_ylabel('Files after dedup/quarantine')
fig.tight_layout();fig.savefig(DIRS['figures']/'fig1_corrected_dataset_distribution.png',dpi=220);plt.close(fig)
fig,ax=plt.subplots(figsize=(9,4));counts.plot.bar(ax=ax,rot=25)
ax.set_title('Frozen real-data group-aware train/validation/test counts');ax.set_ylabel('Images')
fig.tight_layout();fig.savefig(DIRS['figures']/'fig4_real_split.png',dpi=220);plt.close(fig)

## 5 · Independent EfficientNet similarity check

Extracts a second set of ImageNet embeddings using **EfficientNetV2-B0**, independent of the ResNet-50 grouping features. For each validation/test image it finds the nearest training image and summarizes cosine similarity at **0.90** and **0.95**. The flagged pairs provide concrete candidates for review; a high feature similarity is not, by itself, proof of a duplicate.

**Outputs:** `audits/efficientnet_similarity_audit.csv`, `audits/efficientnet_flagged_pairs.csv`, and `audits/leakage_audit_summary.json`.


In [ ]:
logger.info('Calculating independent EfficientNetV2-B0 audit embeddings')
A,B=normalized_embeddings(*embed_images('efficientnet'))
# embeddings correspond to full manifest rows; clean carries same original paths.
map_idx={p:i for i,p in enumerate(manifest.relative_path)}
idx_by_split={s:np.array([map_idx[p] for p in clean.loc[clean.split==s,'relative_path']],dtype=int)
              for s in ('train','val','test')}
summary=[]; suspicious_links=[]
for query in ('val','test'):
    qix=idx_by_split[query]; rix=idx_by_split['train']
    if len(qix)==0 or len(rix)==0: continue
    vals=[]; best_idx=[]
    for offset in range(0,len(qix),CHUNK):
        qi=qix[offset:offset+CHUNK]
        sim=np.maximum(A[qi]@A[rix].T,A[qi]@B[rix].T)
        argmax=sim.argmax(axis=1);vals.extend(sim[np.arange(len(qi)),argmax]);best_idx.extend(rix[argmax])
    vals=np.asarray(vals);best_idx=np.asarray(best_idx)
    for th in (0.90,0.95):
        summary.append({'query_split':query,'reference_split':'train','threshold':th,
                        'queries':len(qix),'count_above_threshold':int((vals>=th).sum()),
                        'share_above_threshold':float((vals>=th).mean()),
                        'median_best_similarity':float(np.median(vals))})
    high=np.flatnonzero(vals>=0.95)
    for j in high:
        suspicious_links.append({'query_split':query,'query_path':manifest.relative_path.iloc[qix[j]],
            'nearest_train_path':manifest.relative_path.iloc[best_idx[j]],'similarity':float(vals[j])})

audit_df=pd.DataFrame(summary);audit_df.to_csv(DIRS['audits']/'efficientnet_similarity_audit.csv',index=False)
pd.DataFrame(suspicious_links,columns=['query_split','query_path','nearest_train_path','similarity']).to_csv(
    DIRS['audits']/'efficientnet_flagged_pairs.csv',index=False)
json_save({'path_overlap':0,'exact_sha256_overlap':0,'phash_group_overlap':0,
           'independent_similarity_results':summary,
           'interpretation':'feature similarity flags are NOT a proof of duplicate identity; manually inspect high matches'},
          DIRS['audits']/'leakage_audit_summary.json')
print(audit_df.to_string(index=False));logger.info('%s high-similarity pairs >=0.95 require review',len(suspicious_links))
del A,B;gc.collect()

## 6 · Image-source shortcut controls

Tests whether six-class labels can be predicted from acquisition-related cues: metadata alone, metadata plus border summaries, or border pixels with the center hidden. The controls share the same frozen real train and test sets. They estimate possible shortcuts in the **disease classifier**, not real-versus-synthetic source accuracy; that separate comparison belongs in E3.

**Output:** `audits/shortcut_control_results.csv`.


In [ ]:
control_rows=[]
if RUN_SHORTCUT_CONTROLS:
    tr=clean[clean.split=='train']; te=clean[clean.split=='test']
    meta_cols=['width','height','aspect_ratio','size_bytes','pixel_std','dark_fraction']
    full=clean.set_index('relative_path')
    for label,feature_cols in [('metadata_only',meta_cols),
        ('metadata_plus_border_summary',meta_cols+[f'border_{s}_{k}' for s in ('mean','std') for k in 'rgb'])]:
        clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1200,class_weight='balanced'))
        clf.fit(tr[feature_cols].to_numpy(),tr.label.to_numpy())
        pred=clf.predict(te[feature_cols].to_numpy())
        control_rows.append({'control':label,'target':'disease_class','test_n':len(te),
            'accuracy':float(accuracy_score(te.label,pred)),
            'macro_f1':float(f1_score(te.label,pred,average='macro',zero_division=0))})
    # Border-only image features: 20x20 thumbnail with CENTER wiped, no lesion detail visible by design.
    def border_features(path):
        arr=load_rgb(path,48)/255.0
        arr[8:-8,8:-8,:]=0.0
        return arr[::3,::3].reshape(-1)
    feat=np.stack([border_features(p) for p in clean.path])
    ix_tr=np.flatnonzero(clean.split.to_numpy()=='train')
    ix_te=np.flatnonzero(clean.split.to_numpy()=='test')
    clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1200,class_weight='balanced'))
    clf.fit(feat[ix_tr],clean.label.to_numpy()[ix_tr])
    pred=clf.predict(feat[ix_te])
    control_rows.append({'control':'border_only_48px','target':'disease_class','test_n':len(ix_te),
        'accuracy':float(accuracy_score(clean.label.to_numpy()[ix_te],pred)),
        'macro_f1':float(f1_score(clean.label.to_numpy()[ix_te],pred,average='macro',zero_division=0))})
    del feat,clf;gc.collect()
else:
    control_rows=[{'control':'shortcut_controls','target':'disease_class','status':'not_run'}]
controls=pd.DataFrame(control_rows)
controls.to_csv(DIRS['audits']/'shortcut_control_results.csv',index=False)
print(controls.to_string(index=False))
json_save({'real_vs_synthetic_source_classifier':'NOT RUN: no synthetic dataset in E2',
    'patient_ids':'Only verified optional metadata, never guessed',
    'interpretation':'Shortcut controls test class predictability from nonlesion features, not medical validity'},
    DIRS['audits']/'pending_e3_controls.json')

## 7 · Preprocessing, batches, and class-weighted focal loss

Decodes RGB images with EXIF orientation correction and **aspect-ratio-preserving padding** to 224 × 224. Only training images receive mild augmentation. Training examples carry per-class sample weights; the focal loss returns one loss value per example so these weights can affect optimization. Validation and test inputs are not augmented or resampled.

**Outputs:** `training/training_class_weights.csv` and a **private** sample-image contact sheet under `private_review/`. The code includes a simple weighted-loss sanity test.


In [ ]:
train_df=clean.loc[clean.split=='train'].reset_index(drop=True)
val_df=clean.loc[clean.split=='val'].reset_index(drop=True)
test_df=clean.loc[clean.split=='test'].reset_index(drop=True)
NUM_CLASSES=len(CLASSES)
train_counts=np.bincount(train_df.label,minlength=NUM_CLASSES)
class_weights=np.asarray([len(train_df)/(NUM_CLASSES*max(1,c)) for c in train_counts],dtype=np.float32)
pd.DataFrame({'class':CLASSES,'train_count':train_counts,'loss_weight':class_weights}).to_csv(
    DIRS['training']/'training_class_weights.csv',index=False)
print('Training class weights:', dict(zip(CLASSES,class_weights.round(3))))

augment_layer=keras.Sequential([
    layers.RandomFlip('horizontal',seed=SEED),
    layers.RandomRotation(0.05,fill_mode='constant',fill_value=127.0,seed=SEED),
    layers.RandomZoom(0.05,fill_mode='constant',fill_value=127.0,seed=SEED),
    layers.RandomContrast(0.08,seed=SEED),
],name='training_augmentation')

def decode_py(path_bytes):
    if hasattr(path_bytes,'numpy'): path_bytes=path_bytes.numpy()
    if isinstance(path_bytes,np.ndarray): path_bytes=path_bytes.item()
    p=path_bytes.decode('utf-8') if isinstance(path_bytes,(bytes,np.bytes_)) else str(path_bytes)
    return load_rgb(p,IMG_SIZE)

def tf_decode(path,label):
    image=tf.numpy_function(lambda p: decode_py(p),[path],tf.float32)
    image.set_shape((IMG_SIZE,IMG_SIZE,3))
    return image,tf.cast(label,tf.int32)

def make_dataset(df,training=False):
    ds=tf.data.Dataset.from_tensor_slices((df.path.astype(str).tolist(),df.label.astype(np.int32).to_numpy()))
    if training: ds=ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    ds=ds.map(tf_decode,num_parallel_calls=4)
    if training:
        def add_weight(im,y):
            im=augment_layer(im,training=True)
            im=tf.clip_by_value(im,0.,255.)
            w=tf.gather(tf.constant(class_weights,dtype=tf.float32),y)
            return im,y,w
        ds=ds.map(add_weight,num_parallel_calls=4)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_dataset(train_df,True)
val_ds=make_dataset(val_df)
test_ds=make_dataset(test_df)

@keras.utils.register_keras_serializable(package='DermReal')
class FocalLossPerExample(keras.losses.Loss):
    def __init__(self,gamma=2.0,alpha=0.25,name='focal_per_example',**kwargs):
        super().__init__(name=name,**kwargs)
        self.gamma=float(gamma);self.alpha=float(alpha)
    def call(self,y_true,y_pred):
        y_true=tf.reshape(tf.cast(y_true,tf.int32),[-1])
        prob=tf.clip_by_value(tf.cast(y_pred,tf.float32),1e-7,1.-1e-7)
        p=tf.gather(prob,y_true,axis=1,batch_dims=1)
        return -self.alpha*tf.pow(1.0-p,self.gamma)*tf.math.log(p)  # shape (batch,)
    def get_config(self):return {**super().get_config(),'gamma':self.gamma,'alpha':self.alpha}

# Check (1) per-example loss shape and (2) explicit weights have a nonzero effect.
_dummy_y=tf.constant([0,1],tf.int32);_dummy_p=tf.constant([[.8,.2],[.7,.3]],tf.float32)
_loss=FocalLossPerExample();_parts=_loss.call(_dummy_y,_dummy_p)
assert _parts.shape==(2,)
assert abs(float(tf.reduce_mean(_parts))-float(tf.reduce_sum(_parts*tf.constant([1.,3.]))/2))>1e-6
print('Per-example weighted focal-loss sanity tests passed.')

# Actual Keras preflight using one decoded training and validation batch, before large backbones.
_x,_y,_w=next(iter(train_ds.take(1)))
_xval,_yval=next(iter(val_ds.take(1)))
assert _x.shape[1:]==(IMG_SIZE,IMG_SIZE,3) and len(_y)==len(_w)==len(_x)
assert np.isfinite(_x.numpy()).all() and np.isfinite(_w.numpy()).all()
assert _xval.shape[1:]==(IMG_SIZE,IMG_SIZE,3)
_small=keras.Sequential([keras.Input(shape=(IMG_SIZE,IMG_SIZE,3)),
    layers.Rescaling(1./255),layers.GlobalAveragePooling2D(),
    layers.Dense(NUM_CLASSES,activation='softmax')])
_small.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=FocalLossPerExample(gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA))
_small_loss=_small.train_on_batch(_x[:2],_y[:2],sample_weight=_w[:2])
assert np.isfinite(float(_small_loss)), 'Tiny weighted training preflight failed'
logger.info('Preflight: decoded train/val batches and weighted Keras training PASS, mini loss=%g',float(_small_loss))
del _small,_x,_y,_w,_xval,_yval; gc.collect()

fig,axes=plt.subplots(2,3,figsize=(12,7))
for ax,cls in zip(axes.flat,CLASSES):
    candidate=train_df.loc[train_df['class']==cls,'path'].iloc[0]
    ax.imshow(load_rgb(candidate)/255.0);ax.set_title(cls);ax.axis('off')
fig.suptitle('REAL-only examples — source/diagnosis must be independently reviewed')
fig.tight_layout();fig.savefig(PRIVATE_REVIEW_DIR/'dataset_contact_sheet.png',dpi=170);plt.close(fig)

## 8 · Model definitions and three-stage fine-tuning

Builds four ImageNet-pretrained convolutional backbones with model-specific preprocessing and a shared six-class classification head. Each model trains through head-only, partial-unfreeze, and deeper-partial-unfreeze phases. Batch-normalization moving statistics stay fixed within pretrained backbones. A callback measures validation **macro-F1**, which controls checkpointing and early stopping; ReduceLROnPlateau is the only learning-rate scheduler.

**Outputs per model:** `models/<model>_p1_best.keras`, `_p2_best.keras`, `_p3_best.keras`, `<model>_final.keras`; `training/<model>_p*_epochs.csv`, `<model>_phase_summary.csv`, and `<model>_selected_checkpoint.json`.


In [ ]:
@keras.utils.register_keras_serializable(package='DermReal')
class ResNetPreprocess(layers.Layer):
    def call(self,x):return keras.applications.resnet50.preprocess_input(tf.cast(x,tf.float32))

@keras.utils.register_keras_serializable(package='DermReal')
class DenseNetPreprocess(layers.Layer):
    def call(self,x):return keras.applications.densenet.preprocess_input(tf.cast(x,tf.float32))

CUSTOM_OBJECTS={'FocalLossPerExample':FocalLossPerExample,
                'ResNetPreprocess':ResNetPreprocess,'DenseNetPreprocess':DenseNetPreprocess}
MODEL_SPECS={
    'EfficientNetV2B0': (EfficientNetV2B0,50,100),
    'ResNet50':(ResNet50,50,100),
    'DenseNet121':(DenseNet121,60,120),
    'MobileNetV3Large':(MobileNetV3Large,25,60),
}

def build_backbone(name):
    cls,_,_=MODEL_SPECS[name]
    kwargs=dict(include_top=False,weights='imagenet',input_shape=(IMG_SIZE,IMG_SIZE,3))
    if name in ['EfficientNetV2B0','MobileNetV3Large']:kwargs['include_preprocessing']=True
    return cls(**kwargs)

def make_classifier(name,backbone):
    inp=keras.Input(shape=(IMG_SIZE,IMG_SIZE,3),name='rgb_0_255')
    x=inp
    if name=='ResNet50':x=ResNetPreprocess(name='resnet_preprocess')(x)
    if name=='DenseNet121':x=DenseNetPreprocess(name='densenet_preprocess')(x)
    x=backbone(x,training=False)   # frozen BatchNorm moving stats even in fine-tuning
    x=layers.GlobalAveragePooling2D()(x)
    x=layers.BatchNormalization()(x)
    x=layers.Dense(512,activation='relu',kernel_regularizer=keras.regularizers.l2(L2_REG))(x)
    x=layers.Dropout(DROPOUT)(x)
    x=layers.Dense(256,activation='relu',kernel_regularizer=keras.regularizers.l2(L2_REG))(x)
    x=layers.Dropout(DROPOUT/2)(x)
    out=layers.Dense(NUM_CLASSES,activation='softmax',dtype='float32')(x)
    return keras.Model(inp,out,name=name)

def configure_trainable(backbone,n_last):
    backbone.trainable=True
    if n_last==0: backbone.trainable=False
    else:
        for layer in backbone.layers[:-n_last]: layer.trainable=False
        for layer in backbone.layers[-n_last:]: layer.trainable=not isinstance(layer,layers.BatchNormalization)
    return sum(int(l.trainable) for l in backbone.layers)

Y_VAL=val_df.label.to_numpy(); Y_TEST=test_df.label.to_numpy()
class ValidationMacroF1(keras.callbacks.Callback):
    def on_epoch_end(self,epoch,logs=None):
        logs=logs if logs is not None else {}
        prob=self.model.predict(val_ds,verbose=0)
        score=f1_score(Y_VAL,prob.argmax(axis=1),average='macro',labels=range(NUM_CLASSES),zero_division=0)
        logs['val_macro_f1']=float(score)
        print(' - val_macro_f1: %.4f'%score)

def compile_phase(model,phase,lr):
    optimizer=keras.optimizers.AdamW(learning_rate=lr,weight_decay=1e-5) if phase==3 else keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer,loss=FocalLossPerExample(gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])

def phase_callbacks(name,phase):
    path=DIRS['models']/f'{name}_p{phase}_best.keras'
    callbacks=[
        ValidationMacroF1(),
        keras.callbacks.ModelCheckpoint(str(path),monitor='val_macro_f1',mode='max',save_best_only=True,verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_macro_f1',mode='max',patience=(5 if phase==1 else 7),
                                     restore_best_weights=True,verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss',mode='min',patience=3,factor=0.5,min_lr=1e-7,verbose=1),
        keras.callbacks.CSVLogger(str(DIRS['training']/f'{name}_p{phase}_epochs.csv'))]
    return callbacks,path

# No contradictory cosine LearningRateScheduler: ReduceLROnPlateau is the single LR policy.
def train_one_model(name):
    keras.backend.clear_session(); gc.collect()
    base=build_backbone(name)
    model=make_classifier(name,base)
    phases=[(1,0,P1_EPOCHS,LR_BY_PHASE[0]),(2,MODEL_SPECS[name][1],P2_EPOCHS,LR_BY_PHASE[1]),
            (3,MODEL_SPECS[name][2],P3_EPOCHS,LR_BY_PHASE[2])]
    phase_rows=[]
    for phase,n_unfrozen,epochs,lr in phases:
        n_train=configure_trainable(base,n_unfrozen)
        compile_phase(model,phase,lr)
        cbs,ckpt=phase_callbacks(name,phase)
        logger.info('%s phase %d started: %d layers trainable, lr=%g',name,phase,n_train,lr)
        hist=model.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=cbs,verbose=1)
        # Ensure the best epoch's weights, not the last epoch, are used for the next phase.
        assert ckpt.is_file(),f'Missing checkpoint: {ckpt}'
        model.load_weights(str(ckpt))
        scores=hist.history['val_macro_f1']; best_score=float(max(scores));best_ep=int(np.argmax(scores))+1
        phase_rows.append({'model':name,'phase':phase,'unfrozen_layers':n_train,'best_epoch':best_ep,
            'epochs_executed':len(scores),'best_val_macro_f1':best_score,'checkpoint':str(ckpt)})
        pd.DataFrame(phase_rows).to_csv(DIRS['training']/f'{name}_phase_summary.csv',index=False)
        logger.info('%s phase %d best validation macro-F1 %.4f',name,phase,best_score)
    # Final checkpoint selection: validation only (test has never been evaluated here).
    selected=max(phase_rows,key=lambda x:x['best_val_macro_f1'])
    final=DIRS['models']/f'{name}_final.keras'
    shutil.copy2(selected['checkpoint'],final)
    json_save({'model':name,'selected_phase':selected['phase'],
        'selection_criterion':'best validation macro-F1 across all phases',
        'validation_f1':selected['best_val_macro_f1'],'checkpoint':str(final)},
        DIRS['training']/f'{name}_selected_checkpoint.json')
    del model,base;gc.collect();keras.backend.clear_session()
    return str(final)

# Lightweight serialization preflight: exercises both custom preprocessing layers and checkpoint loading.
for smoke_name in ('ResNet50','DenseNet121'):
    small_base=keras.Sequential([keras.Input(shape=(IMG_SIZE,IMG_SIZE,3)),
                                 layers.Conv2D(4,3,strides=4,activation='relu')],name='small_backbone')
    smoke_model=make_classifier(smoke_name,small_base)
    smoke_path=DIRS['config']/f'_preflight_{smoke_name}.keras'
    smoke_model.save(str(smoke_path))
    restored=keras.models.load_model(str(smoke_path),custom_objects=CUSTOM_OBJECTS,compile=False)
    prob=restored.predict(np.zeros((1,IMG_SIZE,IMG_SIZE,3),dtype=np.float32),verbose=0)
    assert prob.shape==(1,NUM_CLASSES) and np.isfinite(prob).all()
    smoke_path.unlink(missing_ok=True)
    del small_base,smoke_model,restored;keras.backend.clear_session();gc.collect()
print('Model builders, checkpoint serialization and validation macro-F1 callbacks ready.')

## 9 · Metric computation and scientific figures

Calculates test accuracy, balanced accuracy, macro precision, macro recall, macro/weighted F1, ROC-AUC, Cohen's kappa, and Matthews correlation coefficient. Also saves per-class precision/recall/F1/support, confusion-matrix counts, and **per-image class probabilities** so the scores can be recomputed independently. Figures are regenerated from the new real dataset rather than copied from the old experiment.

**Outputs:** `metrics/<model>_{val,test}_predictions.csv`, `<model>_{val,test}_per_class.csv`, `<model>_{val,test}_confusion_counts.csv` and `figures/<model>_{confusion_matrix,ROC}.png`.


In [ ]:
METRIC_ROWS=[]
ALL_PROBS={}

def compute_metrics(y,prob,name,split='test'):
    y=np.asarray(y,dtype=int);p=np.asarray(prob,dtype=np.float32)
    assert p.ndim==2 and p.shape==(len(y),NUM_CLASSES),f'Bad prediction shape for {name}: {p.shape}'
    yh=p.argmax(axis=1)
    pr,rc,fm,support=precision_recall_fscore_support(y,yh,labels=list(range(NUM_CLASSES)),zero_division=0)
    try:
        yt=label_binarize(y,classes=np.arange(NUM_CLASSES))
        roc=float(roc_auc_score(yt,p,average='macro',multi_class='ovr'))
    except ValueError:roc=float('nan')
    row={'Model':name,'split':split,'n':len(y),'Accuracy':float(accuracy_score(y,yh)),
         'Balanced Accuracy':float(balanced_accuracy_score(y,yh)),
         'Precision (Macro)':float(np.mean(pr)),'Recall (Macro)':float(np.mean(rc)),
         'F1 (Macro)':float(f1_score(y,yh,labels=range(NUM_CLASSES),average='macro',zero_division=0)),
         'F1 (Weighted)':float(f1_score(y,yh,labels=range(NUM_CLASSES),average='weighted',zero_division=0)),
         'AUC-ROC':roc,'Cohen Kappa':float(cohen_kappa_score(y,yh)),
         'MCC':float(matthews_corrcoef(y,yh))}
    cm=confusion_matrix(y,yh,labels=range(NUM_CLASSES)); pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(
        DIRS['metrics']/f'{name}_{split}_confusion_counts.csv')
    pd.DataFrame({'class':CLASSES,'precision':pr,'recall':rc,'f1':fm,'support':support}).to_csv(
        DIRS['metrics']/f'{name}_{split}_per_class.csv',index=False)
    df_pred=pd.DataFrame({'relative_path':(test_df if split=='test' else val_df).relative_path,
                           'sha256':(test_df if split=='test' else val_df).sha256,
                           'true_label':y,'true_class':np.array(CLASSES)[y],
                           'predicted_label':yh,'predicted_class':np.array(CLASSES)[yh]})
    for i,cls in enumerate(CLASSES):df_pred[f'prob_{cls}']=p[:,i]
    df_pred.to_csv(DIRS['metrics']/f'{name}_{split}_predictions.csv',index=False)
    if split=='test':
        fig,axes=plt.subplots(1,2,figsize=(15,6))
        sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=CLASSES,yticklabels=CLASSES,ax=axes[0])
        sns.heatmap(cm/np.maximum(cm.sum(axis=1,keepdims=True),1),annot=True,fmt='.2f',cmap='Greens',
                    xticklabels=CLASSES,yticklabels=CLASSES,ax=axes[1])
        for ax in axes:ax.set_xlabel('Predicted');ax.set_ylabel('True');ax.tick_params(axis='x',rotation=40)
        fig.suptitle(name+' — real test confusion matrices')
        fig.tight_layout();fig.savefig(DIRS['figures']/f'{name}_confusion_matrix.png',dpi=220,bbox_inches='tight');plt.close(fig)
        fig,ax=plt.subplots(figsize=(8,6));yt=label_binarize(y,classes=np.arange(NUM_CLASSES))
        for i,cls in enumerate(CLASSES):
            if yt[:,i].min()==yt[:,i].max():continue
            fpr,tpr,_=roc_curve(yt[:,i],p[:,i]);ax.plot(fpr,tpr,label=cls)
        ax.plot([0,1],[0,1],'k--',linewidth=1);ax.legend(fontsize=8)
        ax.set(xlabel='False positive rate',ylabel='True positive rate',title=name+' — real test one-vs-rest ROC')
        fig.tight_layout();fig.savefig(DIRS['figures']/f'{name}_ROC.png',dpi=220);plt.close(fig)
    return row

def model_predictions(model,ds):
    p=model.predict(ds,verbose=0)
    return np.asarray(p,dtype=np.float32)

print('Evaluation tools ready.')

## 10 · Train and evaluate the four CNNs

**GPU-intensive section.** Runs EfficientNetV2-B0, ResNet-50, DenseNet-121, and MobileNetV3-Large in sequence. Each model's best checkpoint is reloaded for validation and real-test predictions, and the growing leaderboard is saved after each completed model.

**Outputs:** `metrics/model_comparison.csv` and the model-specific checkpoints, training logs, prediction CSVs, and figures described above. The run log records model starts and completed evaluations.


In [ ]:
TRAINED_MODELS=[]
for model_name in RUN_MODELS:
    assert model_name in MODEL_SPECS,model_name
    logger.info('=== START %s ===',model_name)
    final_path=train_one_model(model_name)
    model=keras.models.load_model(final_path,custom_objects=CUSTOM_OBJECTS,compile=False)
    val_prob=model_predictions(model,val_ds)
    test_prob=model_predictions(model,test_ds)
    selected_val=compute_metrics(Y_VAL,val_prob,model_name,'val')
    row=compute_metrics(Y_TEST,test_prob,model_name,'test')
    ALL_PROBS[model_name]={'val':val_prob,'test':test_prob}
    METRIC_ROWS.append(row);TRAINED_MODELS.append(model_name)
    pd.DataFrame(METRIC_ROWS).to_csv(DIRS['metrics']/'model_comparison.csv',index=False)
    logger.info('%s validation F1=%.4f; real TEST accuracy=%.4f macroF1=%.4f',
                model_name,selected_val['F1 (Macro)'],row['Accuracy'],row['F1 (Macro)'])
    del model,val_prob,test_prob;gc.collect();keras.backend.clear_session()
print(pd.DataFrame(METRIC_ROWS).sort_values('F1 (Macro)',ascending=False).to_string(index=False))

## 11 · Validation-weighted ensemble and final metrics

Constructs soft-voting ensemble weights from the four models' **validation macro-F1** values, then scores the fixed real-only test set once. Writes ensemble predictions alongside each individual model's predictions and creates a results dashboard. The `per_source_*` tables here contain the **real-only** slice; a synthetic slice will be created by E3.

**Outputs:** `ensemble/ensemble_weights.csv`, `metrics/model_comparison_with_ensemble.csv`, `metrics/final_leaderboard.csv`, `metrics/per_source_overall.csv`, `metrics/per_source_recall.csv`, `figures/model_comparison_dashboard.png`, plus ensemble confusion matrices, class metrics and prediction files.


In [ ]:
assert len(TRAINED_MODELS)>=1,'No completed models'
ensemble_weights=[]
if len(TRAINED_MODELS)>=2:
    val_f1=np.array([f1_score(Y_VAL,ALL_PROBS[n]['val'].argmax(axis=1),
                  labels=range(NUM_CLASSES),average='macro',zero_division=0) for n in TRAINED_MODELS])
    weights=val_f1/val_f1.sum() if val_f1.sum()>0 else np.full(len(TRAINED_MODELS),1/len(TRAINED_MODELS))
    ensemble_weights=[{'model':n,'validation_macro_f1':float(f),'weight':float(w)} for n,f,w in zip(TRAINED_MODELS,val_f1,weights)]
    pd.DataFrame(ensemble_weights).to_csv(DIRS['ensemble']/'ensemble_weights.csv',index=False)
    ens_val=sum(float(w)*ALL_PROBS[n]['val'] for w,n in zip(weights,TRAINED_MODELS))
    ens_test=sum(float(w)*ALL_PROBS[n]['test'] for w,n in zip(weights,TRAINED_MODELS))
    _=compute_metrics(Y_VAL,ens_val,'Ensemble_Weighted','val')
    ens_metrics=compute_metrics(Y_TEST,ens_test,'Ensemble_Weighted','test')
    METRIC_ROWS.append(ens_metrics)
    pd.DataFrame(METRIC_ROWS).to_csv(DIRS['metrics']/'model_comparison_with_ensemble.csv',index=False)
    logger.info('Ensemble TEST accuracy=%.4f, macro-F1=%.4f; weights learned ONLY on val',
                ens_metrics['Accuracy'],ens_metrics['F1 (Macro)'])
else:
    pd.DataFrame(columns=['model','validation_macro_f1','weight']).to_csv(
        DIRS['ensemble']/'ensemble_weights.csv',index=False)
    logger.warning('Ensemble skipped because fewer than 2 models completed')

pd.DataFrame([{'model':r['Model'],'source':'real','test_n':r['n'],
               'accuracy':r['Accuracy'],'precision_macro':r['Precision (Macro)'],
               'recall_macro':r['Recall (Macro)'],'macro_f1':r['F1 (Macro)']} for r in METRIC_ROWS]).to_csv(
    DIRS['metrics']/'per_source_overall.csv',index=False)
recall_records=[]
for r in METRIC_ROWS:
    file=DIRS['metrics']/f"{r['Model']}_test_per_class.csv"
    if file.exists():
        df=pd.read_csv(file)
        for _,item in df.iterrows(): recall_records.append({'model':r['Model'],'source':'real',
            'class':item['class'],'precision':item['precision'],'recall':item['recall'],
            'f1':item['f1'],'support':item['support']})
pd.DataFrame(recall_records).to_csv(DIRS['metrics']/'per_source_recall.csv',index=False)

all_metrics=pd.DataFrame(METRIC_ROWS).sort_values('F1 (Macro)',ascending=False)
fig,ax=plt.subplots(figsize=(10,5));all_metrics.set_index('Model')[['Accuracy','F1 (Macro)','Recall (Macro)']].plot.bar(ax=ax,rot=20)
ax.set(ylim=(0,1),title='Corrected real-only test: accuracy, macro-F1, macro-recall')
fig.tight_layout();fig.savefig(DIRS['figures']/'model_comparison_dashboard.png',dpi=220);plt.close(fig)
all_metrics.to_csv(DIRS['metrics']/'final_leaderboard.csv',index=False)
print(all_metrics.to_string(index=False))

## 12 · Export results for research and GitHub

Generates an experiment summary, **artifact inventory**, and one ZIP containing code-independent evidence: configuration, hashes/group statistics, frozen splits, audits, training histories, metric tables, and publication figures. The archive excludes raw photographs, private image-review galleries, and large checkpoints.

**Outputs:** `experiment_summary.json`, `artifact_inventory.csv`, `artifact_validation.json`, and `/kaggle/working/dermareal_e2_evidence.zip` (when running on Kaggle).

Keep the `models/` checkpoints separately in Kaggle outputs or another versioned artifact store. The split file `splits/splits_v3.csv` is the input to E3.


In [ ]:
import zipfile
summary={'experiment':'E2_REAL_ONLY','dataset_fingerprint':dataset_fingerprint,
         'raw_file_count':int(N),'eligible_count':len(clean),'class_names':CLASSES,
         'groups':int(clean.group.nunique()),'split_counts':clean.split.value_counts().to_dict(),
         'training_finished_models':TRAINED_MODELS,'full_gpu_training_executed':len(TRAINED_MODELS)==4,
         'checkpoints_are_in':'models/*.keras (not included in GitHub evidence zip)',
         'label_review':'Requires manually verified source diagnosis and licences',
         'patient_independence':'Not proven without actual patient IDs',
         'held_out_source':'real only','pending_for_E3':['real/synthetic source-classifier',
             'same frozen real-test-set comparison','synthetic-test-only metrics','fixed mixed-training pool']}
json_save(summary,ROOT/'experiment_summary.json')
# Machine-readable inventory and checks: no claimed artifact is silently omitted.
required_files=[
    'config/run_config.json','config/environment.json',
    'dataset/dataset_manifest.csv','dataset/dataset_statistics.csv','dataset/dataset_fingerprint.json',
    'dataset/validation_summary.json','dataset/provenance_excluded.csv',
    'audits/exact_sha256_groups.csv','audits/exact_md5_groups.csv','audits/group_summary.csv',
    'audits/group_membership.csv','audits/removals_by_class_reason.csv',
    'audits/removals_by_source_reason.csv','audits/efficientnet_similarity_audit.csv',
    'audits/efficientnet_flagged_pairs.csv','audits/leakage_audit_summary.json',
    'audits/shortcut_control_results.csv','splits/splits_v3.csv',
    'splits/split_class_counts.csv','splits/split_group_counts.csv',
    'training/training_class_weights.csv','metrics/model_comparison.csv',
    'metrics/per_source_overall.csv','metrics/per_source_recall.csv','metrics/final_leaderboard.csv',
    'ensemble/ensemble_weights.csv','figures/fig1_corrected_dataset_distribution.png',
    'figures/fig4_real_split.png','figures/model_comparison_dashboard.png','logs/run.log'
]
if len(TRAINED_MODELS)>=2:
    required_files.append('metrics/model_comparison_with_ensemble.csv')
for name in TRAINED_MODELS:
    required_files += [
        f'training/{name}_phase_summary.csv',f'training/{name}_selected_checkpoint.json',
        f'metrics/{name}_test_predictions.csv',f'metrics/{name}_test_per_class.csv',
        f'metrics/{name}_test_confusion_counts.csv',
    ]
required_files=sorted(set(required_files))
missing=[name for name in required_files if not (ROOT/name).is_file()]
json_save({'total_expected':len(required_files),'missing':missing,'missing_count':len(missing),
           'completed_models':TRAINED_MODELS}, ROOT/'artifact_validation.json')
file_rows=[]
for item in sorted(ROOT.rglob('*')):
    if item.is_file():
        rp=item.relative_to(ROOT).as_posix()
        file_rows.append({'relative_path':rp,'bytes':item.stat().st_size,
                          'include_in_public_evidence':bool('private_review' not in item.relative_to(ROOT).parts
                           and item.suffix.lower() not in {'.keras','.h5','.npz','.npy','.pt','.pth','.ckpt'}),
                          'expected':rp in required_files})
pd.DataFrame(file_rows).to_csv(ROOT/'artifact_inventory.csv',index=False)
if missing: raise RuntimeError(f'Expected experiment outputs missing: {missing}')

zip_path=ROOT.parent/'dermareal_e2_evidence.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as zf:
    for item in sorted(ROOT.rglob('*')):
        if item.is_file() and 'private_review' not in item.relative_to(ROOT).parts and item.suffix.lower() not in {'.keras','.h5','.npz','.npy','.pt','.pth','.ckpt'}:
            zf.write(item,arcname=str(item.relative_to(ROOT)))
print('Evidence ZIP:',zip_path)
print('Full run output:',ROOT)
print('Model checkpoints: ',DIRS['models'])
print('Frozen E3 input: ',DIRS['splits']/'splits_v3.csv')
print('All evidence files:',len(file_rows),'files inventoried; required files:',len(required_files))
for p in sorted(ROOT.rglob('*')):
    if p.is_file() and p.suffix!='.keras': print('  ',p.relative_to(ROOT))